In [ ]:
import os
for key in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[key] = "1"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import sys
import json
import platform
import random
import hashlib
import importlib.util
import importlib.metadata
from pathlib import Path
from uuid import uuid4
from copy import deepcopy

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / "function").is_dir()
             and (p / "example" / "parameters.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run this notebook from the repository root or example directory")
EXAMPLE = ROOT / "example"
sys.path.insert(0, str(ROOT))
spec = importlib.util.spec_from_file_location("real_experiment_parameters", EXAMPLE / "parameters.py")
parameters = importlib.util.module_from_spec(spec)
spec.loader.exec_module(parameters)

import numpy as np
import pandas as pd
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt
import torch
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import PercentFormatter, MultipleLocator, FormatStrFormatter
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.offsetbox import DrawingArea, TextArea, HPacker, VPacker, AnchoredOffsetbox
from sklearn.metrics import roc_curve, roc_auc_score, auc
from threadpoolctl import threadpool_limits
from IPython.display import display
from function.real_data import (
    load_data, training_scale, save_json, sha256, fit_full, find_rscript,
    predict_linear, scores, ridge_fit, bh_adjust, gene_statistics,
)

random.seed(parameters.SEED)
np.random.seed(parameters.SEED)
torch.manual_seed(parameters.SEED)
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
threadpool_limits(limits=1)
METHODS = ["LR", "Lasso", "Elastic Net", "SCAD", "DWR", "SRDO", "SVI", "VBLR", "DW-BLR"]
TOP_K = (3, 5, 10)
DATA = EXAMPLE / "data" / "data.xlsx"
TOP_GENES_FILE = EXAMPLE / "data" / "top_genes.json"
SUBGROUP_FILE = EXAMPLE / "data" / "clinical_subgroups.csv"
SUBGROUP_PROVENANCE_FILE = EXAMPLE / "data" / "clinical_subgroups_provenance.json"
REUSE_MATCHING_CACHE = True
REFIT_METHODS = set()


In [ ]:
def load_top_gene_sets(path, methods, top_k, available_genes):
    contents = json.loads(Path(path).read_text(encoding="utf-8-sig"))
    if not isinstance(contents, dict) or set(contents) != set(methods):
        raise ValueError("top_genes.json must specify exactly the configured methods")
    expected_settings = {f"Top{k}" for k in top_k}
    available = set(available_genes)
    result = {}
    for method in methods:
        block = contents[method]
        if not isinstance(block, dict) or set(block) != expected_settings:
            raise ValueError(f"Missing or unexpected Top-K lists: {method}")
        for k in top_k:
            names = block[f"Top{k}"]
            if not isinstance(names, list) or not all(isinstance(g, str) and g for g in names):
                raise ValueError(f"Gene names must be nonempty strings: {method}, Top{k}")
            if len(names) != k or len(set(names)) != k:
                raise ValueError(f"Expected exactly {k} distinct genes: {method}, Top{k}")
            missing = set(names) - available
            if missing:
                raise ValueError(f"Genes absent from the expression matrix: {method}, Top{k}: {sorted(missing)}")
            result[(method, k)] = names.copy()
    return result


In [ ]:
def load_clinical_subgroups(path, cohorts):
    clinical = pd.read_csv(path, keep_default_na=False)
    required = {"cohort", "sample_id", "pCR", "subgroup", "ER_status", "PR_status",
                "HER2", "HR", "HR_by_ESR1_fallback"}
    if not required.issubset(clinical.columns) or clinical.duplicated(["cohort", "sample_id"]).any():
        raise ValueError("Missing clinical fields or duplicate cohort/sample IDs")
    if set(clinical.cohort) != set(cohorts):
        raise ValueError("Clinical annotations must cover the supplied cohorts")
    def receptor(values):
        return values.astype(str).str.strip().str.lower().map(
            {"positive": 1, "p": 1, "1": 1, "1.0": 1,
             "negative": 0, "n": 0, "0": 0, "0.0": 0})
    matched = []
    for cohort, data in cohorts.items():
        part = clinical[clinical.cohort.eq(cohort)].set_index("sample_id")
        if set(part.index) != set(data["sample_id"]):
            raise ValueError(f"Clinical sample IDs do not match the expression data: {cohort}")
        part = part.loc[data["sample_id"]].reset_index()
        np.testing.assert_array_equal(pd.to_numeric(part.pCR).to_numpy(), data["y"])
        if not pd.to_numeric(part.HER2).eq(0).all():
            raise ValueError(f"These subgroup definitions require HER2-negative samples: {cohort}")
        er, pr, hr = receptor(part.ER_status), receptor(part.PR_status), receptor(part.HR)
        positive = hr.eq(1) if cohort in ("GSE25066", "GSE194040") else er.eq(1) | pr.eq(1)
        negative = hr.eq(0) if cohort in ("GSE25066", "GSE194040") else er.eq(0) & pr.eq(0)
        expected = np.select([positive, negative], ["HR+/HER2-", "TNBC"], default="Unclassified HR")
        if not np.array_equal(part.subgroup.to_numpy(), expected):
            raise ValueError(f"Subgroup labels disagree with the recorded receptor rules: {cohort}")
        matched.append(part)
    return pd.concat(matched, ignore_index=True)


In [ ]:
cohorts, genes = load_data(DATA, parameters.N_TRAIN, parameters.N_FEATURES)
top_sets = load_top_gene_sets(TOP_GENES_FILE, METHODS, TOP_K, genes)
clinical_provenance = json.loads(SUBGROUP_PROVENANCE_FILE.read_text(encoding="utf-8"))
clinical_content = SUBGROUP_FILE.read_text(encoding="utf-8-sig")
if hashlib.sha256(clinical_content.encode("utf-8")).hexdigest() != clinical_provenance["sha256"]:
    raise ValueError("Clinical subgroup file does not match its bundled provenance")
clinical = load_clinical_subgroups(SUBGROUP_FILE, cohorts)
TRAIN = next(iter(cohorts))
TESTS = list(cohorts)[1:]
X, y = cohorts[TRAIN]["X"], cohorts[TRAIN]["y"]
mean, scale = training_scale(X)
Z = {name: (data["X"] - mean) / scale for name, data in cohorts.items()}
configs = parameters.all_parameters(len(y), len(genes))
configs["LR"] = parameters._internal_weighted_fit()
refit_config = parameters.topk_refit_parameters()
r_configs = {name: configs[name] for name in ("Lasso", "Elastic Net", "SCAD")}
rscript = find_rscript(r_configs)

packages = ["numpy", "scipy", "pandas", "openpyxl", "scikit-learn", "torch", "matplotlib", "nbformat"]
versions = {name: importlib.metadata.version(name) for name in packages}
source_paths = sorted(set([EXAMPLE / "parameters.py"] +
    list((ROOT / "function").glob("*.py")) + list((ROOT / "_backends").glob("*.py")) +
    [ROOT / "function" / "real_data_penalties.R"]))
source_hashes = {str(p.relative_to(ROOT)): sha256(p) for p in source_paths}
fit_identity = dict(data_sha256=sha256(DATA), source_sha256=source_hashes,
                versions=versions)
fit_fingerprint = hashlib.sha256(json.dumps(fit_identity, sort_keys=True).encode()).hexdigest()
NATIVE_MODEL_DIR = EXAMPLE / "results" / ("real_data_" + fit_fingerprint[:12]) / "models"
identity = dict(fit_identity,
    top_genes_sha256=sha256(TOP_GENES_FILE), clinical_subgroups_sha256=sha256(SUBGROUP_FILE))
fingerprint = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()
RUN_DIR = EXAMPLE / "results" / ("real_data_" + fingerprint[:12])
FIG_DIR = RUN_DIR / "figures"
TABLE_DIR = RUN_DIR / "tables"
MODEL_DIR = RUN_DIR / "models"
for path in (FIG_DIR, TABLE_DIR, MODEL_DIR):
    path.mkdir(parents=True, exist_ok=True)
manifest = dict(identity, fingerprint=fingerprint, python=sys.version,
                platform=platform.platform(), Rscript=rscript,
                R_LIBS_USER=os.environ.get("R_LIBS_USER"),
                parameters=configs, topk_refit=refit_config, status="running",
                notebook_sha256=sha256(EXAMPLE / "real_data.ipynb"),
                full_model_cache_fingerprint=fit_fingerprint,
                full_model_cache_directory=str(NATIVE_MODEL_DIR.relative_to(ROOT)),
                top_gene_source=dict(file=str(TOP_GENES_FILE.relative_to(ROOT)),
                    sha256=identity["top_genes_sha256"],
                    lists={m: {f"Top{k}": top_sets[(m, k)] for k in TOP_K} for m in METHODS}))
save_json(RUN_DIR / "manifest.json", manifest)
pd.DataFrame({"gene": genes, "training_mean": mean, "training_sd_ddof0": scale}).to_csv(
    TABLE_DIR / "training_scaler.csv", index=False)
cohort_summary = pd.DataFrame([dict(cohort=c, split="train" if c == TRAIN else "test",
    n=len(d["y"]), pCR=int(d["y"].sum()), RD=int((d["y"] == 0).sum())) for c, d in cohorts.items()])
cohort_summary.to_csv(TABLE_DIR / "cohort_summary.csv", index=False)
display(cohort_summary)


In [ ]:
fits, shared, fit_sources = {}, {}, {}
for method in METHODS:
    directory = NATIVE_MODEL_DIR / method.replace(" ", "_")
    cached = directory / "fit.json"
    print(method, flush=True)
    try:
        if REUSE_MATCHING_CACHE and method not in REFIT_METHODS and cached.is_file():
            fit = json.loads(cached.read_text(encoding="utf-8"))
            fit_sources[method] = "matching_cache"
        else:
            fit_directory = directory
            if method == "DW-BLR" and (directory / "trace").exists():
                fit_directory = directory / "refits" / uuid4().hex
            fit = fit_full(method, X, Z[TRAIN], y, genes, cohorts[TRAIN]["sample_id"],
                           parameters, fit_directory, shared)
            if method == "DW-BLR":
                fit["diagnostics"]["trace_directory"] = str((fit_directory / "trace").relative_to(NATIVE_MODEL_DIR.parent))
            save_json(cached, fit)
            fit_sources[method] = "fitted_this_run"
        if len(fit["beta"]) != len(genes) or not np.isfinite(fit["beta"]).all():
            raise ValueError(f"Invalid coefficients: {method}")
        fits[method] = fit
        save_json(MODEL_DIR / method.replace(" ", "_") / "fit.json", fit)
        if (directory / "failure.json").is_file():
            (directory / "failure.json").replace(directory / "previous_failure.json")
    except Exception as error:
        save_json(directory / "failure.json", {"method": method, "error": repr(error)})
        manifest.update(status="failed", failed_method=method, error=repr(error))
        save_json(RUN_DIR / "manifest.json", manifest)
        raise
assert len(fits) == 9
svi_selected = np.flatnonzero(fits["SVI"]["selected"])
svi_check_cfg = parameters._internal_weighted_fit()
svi_check_cfg["lambda_l2"] = configs["SVI"]["final_l2"]
svi_check = ridge_fit(Z[TRAIN][:, svi_selected], y, svi_check_cfg,
    weights=np.asarray(fits["SVI"]["weights"]) if configs["SVI"]["final_use_srdo_weights"] else None)
svi_difference = np.max(np.abs(predict_linear(svi_check, Z[TRAIN][:, svi_selected])
                              - predict_linear(fits["SVI"], Z[TRAIN])))
assert svi_difference < 1e-4, svi_difference
save_json(MODEL_DIR / "SVI" / "independent_refit_verification.json",
          dict(prediction_max_difference=float(svi_difference), **svi_check["diagnostics"]))
convergence = pd.DataFrame([dict(method=m, converged=f["diagnostics"]["converged"],
    selected=int(np.sum(f["selected"])), elapsed_seconds=f["diagnostics"]["elapsed_seconds"])
    for m, f in fits.items()])
convergence.to_csv(TABLE_DIR / "convergence.csv", index=False)
display(convergence)


In [ ]:
metric_columns = ["Log_loss", "Brier", "AUROC", "AUPRC", "AP"]

def summarize_test_cohorts(frame):
    if not frame.split.eq("test").all():
        raise ValueError("The cross-cohort summary must exclude training observations")
    if set(frame.method) != set(METHODS):
        raise ValueError("All nine methods are required")
    rows = []
    for method in METHODS:
        sub = frame[frame.method.eq(method)]
        if len(sub) != len(TESTS) or sub.cohort.duplicated().any() or set(sub.cohort) != set(TESTS):
            raise ValueError(f"Expected one result per test cohort: {method}")
        if sub.setting.nunique() != 1 or not np.isfinite(sub[metric_columns].to_numpy()).all():
            raise ValueError(f"Mixed model settings or non-finite metrics: {method}")
        row = dict(method=method, setting=sub.setting.iloc[0], n_test_cohorts=len(sub))
        for metric in metric_columns:
            values = sub[metric].to_numpy(dtype=float)
            worst = values.max() if metric in ("Log_loss", "Brier") else values.min()
            row.update({f"{metric}_mean": float(values.mean()),
                        f"{metric}_SD": float(values.std(ddof=1)),
                        f"{metric}_worst": float(worst),
                        f"{metric}_worst_cohort": ";".join(sub.loc[sub[metric].eq(worst), "cohort"])})
        rows.append(row)
    return pd.DataFrame(rows)

def format_performance_table(summary):
    table = summary[["method", "setting", "n_selected_genes", "n_nonzero_prediction_coefficients"]].copy()
    for metric in metric_columns[:4]:
        table[f"{metric}_mean(SD)"] = [f"{m:.3f} ({s:.3f})" for m, s in
            zip(summary[f"{metric}_mean"], summary[f"{metric}_SD"])]
        table[f"{metric}_worst"] = summary[f"{metric}_worst"].map(lambda v: f"{v:.3f}")
    return table

full_metric_rows, full_prediction_frames, full_coefficient_frames, full_model_rows = [], [], [], []
for method in METHODS:
    fit = fits[method]
    beta = np.asarray(fit["beta"], dtype=float)
    selected = np.asarray(fit["selected"], dtype=bool)
    beta_input = beta / scale
    intercept_input = float(fit["intercept"] - mean @ beta_input)
    details = dict(method=method, setting="Full",
        n_input_genes=len(genes), n_selected_genes=int(selected.sum()),
        n_nonzero_prediction_coefficients=int(np.count_nonzero(beta)),
        intercept_standardized=float(fit["intercept"]), intercept_input_scale=intercept_input,
        fit_source=fit_sources[method], converged=bool(fit["diagnostics"]["converged"]))
    full_model_rows.append(details)
    full_coefficient_frames.append(pd.DataFrame(dict(method=method, gene=genes,
        feature_index=np.arange(len(genes)), beta_standardized=beta, beta_input_scale=beta_input,
        selected=selected, nonzero_in_predictor=beta != 0,
        importance=fit["importance"], importance_type=fit["importance_type"])))
    save_json(MODEL_DIR / method.replace(" ", "_") / "full_model.json", dict(
        **details, training_cohort=TRAIN, genes=genes, training_mean=mean,
        training_sd_ddof0=scale, beta_standardized=beta, beta_input_scale=beta_input,
        selected=selected, importance=fit["importance"], importance_type=fit["importance_type"],
        parameters=configs[method], diagnostics=fit["diagnostics"],
        data_sha256=identity["data_sha256"], fit_fingerprint=fit_fingerprint,
        analysis_fingerprint=fingerprint, native_fit_source=str(
            (NATIVE_MODEL_DIR / method.replace(" ", "_") / "fit.json").relative_to(ROOT))))
    for cohort, data in cohorts.items():
        p = predict_linear(fit, Z[cohort])
        input_p = predict_linear(dict(intercept=intercept_input, beta=beta_input), data["X"])
        np.testing.assert_allclose(p, input_p, rtol=0, atol=1e-10)
        full_metric_rows.append(dict(method=method, setting="Full", top_k=None,
            cohort=cohort, split="train_apparent" if cohort == TRAIN else "test",
            n=len(p), pCR=int(data["y"].sum()),
            source_model_converged=bool(fit["diagnostics"]["converged"]),
            refit_converged=bool(fit["diagnostics"]["converged"]), **scores(data["y"], p)))
        full_prediction_frames.append(pd.DataFrame(dict(method=method, setting="Full",
            cohort=cohort, sample_id=data["sample_id"], pCR=data["y"], probability=p)))
full_metrics = pd.DataFrame(full_metric_rows)
full_predictions = pd.concat(full_prediction_frames, ignore_index=True)
full_coefficients = pd.concat(full_coefficient_frames, ignore_index=True)
full_model_details = pd.DataFrame(full_model_rows)
full_summary = summarize_test_cohorts(full_metrics[full_metrics.split.eq("test")]).merge(
    full_model_details, on=["method", "setting"], validate="one_to_one")
for name, frame in {"full_model_metrics": full_metrics, "full_model_predictions": full_predictions,
        "full_model_coefficients": full_coefficients, "full_model_details": full_model_details,
        "full_model_summary": full_summary,
        "full_model_summary_formatted": format_performance_table(full_summary)}.items():
    frame.to_csv(TABLE_DIR / f"{name}.csv", index=False)
FULL_COHORT_DIR = TABLE_DIR / "full_models_by_cohort"
FULL_METHOD_DIR = TABLE_DIR / "full_models_by_method"
for directory in (FULL_COHORT_DIR, FULL_METHOD_DIR):
    directory.mkdir(exist_ok=True)
for cohort in cohorts:
    full_metrics[full_metrics.cohort.eq(cohort)].merge(full_model_details,
        on=["method", "setting"], validate="one_to_one").to_csv(FULL_COHORT_DIR / f"{cohort}.csv", index=False)
for method in METHODS:
    full_metrics[full_metrics.method.eq(method)].to_csv(
        FULL_METHOD_DIR / (method.replace(" ", "_") + ".csv"), index=False)
manifest["full_model_results"] = dict(status="complete", fit_sources=fit_sources,
    n_metric_rows=len(full_metrics), n_prediction_rows=len(full_predictions),
    nonconverged_methods=full_model_details.loc[~full_model_details.converged, "method"].tolist())
save_json(RUN_DIR / "manifest.json", manifest)
display(format_performance_table(full_summary))


In [ ]:
top_fits, top_rows = {}, []
for method in METHODS:
    for k in TOP_K:
        names = top_sets[(method, k)]
        indices = [genes.index(name) for name in names]
        lambdas = refit_config["lambda_by_method"].get(method)
        lam = (lambdas[f"Top{k}"] if lambdas else configs["LR"]["lambda_l2"])
        cfg = deepcopy(refit_config)
        cfg["lambda_l2"] = lam
        if cfg["sample_weights"] != "ones" or cfg["refit_on_test"] or cfg["clinical_covariates"]:
            raise ValueError("Top-K refits require equal weights, no clinical covariates and training-only fitting")
        top_fits[(method, k)] = ridge_fit(Z[TRAIN][:, indices], y, cfg, topk=True)
        save_json(MODEL_DIR / method.replace(" ", "_") / f"Top{k}_ridge.json",
                  dict(top_fits[(method, k)], genes=names, indices=indices,
                       gene_source=TOP_GENES_FILE.name, gene_source_sha256=identity["top_genes_sha256"]))
        for rank, gene in enumerate(names, 1):
            top_rows.append(dict(method=method, top_k=k, rank=rank, gene=gene,
                source_file=TOP_GENES_FILE.name, lambda_l2=lam))
top_gene_table = pd.DataFrame(top_rows)
top_gene_table.to_csv(TABLE_DIR / "top_gene_sets.csv", index=False)
display(top_gene_table[top_gene_table.top_k.eq(10)].pivot(index="rank", columns="method", values="gene"))


In [ ]:
metric_rows, prediction_frames = full_metrics.to_dict("records"), [full_predictions]
for method in METHODS:
    for k in TOP_K:
        setting, fit = f"Top{k}", top_fits[(method, k)]
        indices = [genes.index(g) for g in top_sets[(method, k)]]
        for cohort, data in cohorts.items():
            p = predict_linear(fit, Z[cohort][:, indices])
            metric_rows.append(dict(method=method, setting=setting, top_k=k,
                cohort=cohort, split="train_apparent" if cohort == TRAIN else "test",
                n=len(p), pCR=int(data["y"].sum()),
                source_model_converged=bool(fits[method]["diagnostics"]["converged"]),
                refit_converged=bool(fit["diagnostics"]["converged"]), **scores(data["y"], p)))
            prediction_frames.append(pd.DataFrame(dict(method=method, setting=setting,
                cohort=cohort, sample_id=data["sample_id"], pCR=data["y"], probability=p)))
metrics = pd.DataFrame(metric_rows)
predictions = pd.concat(prediction_frames, ignore_index=True)
metrics.to_csv(TABLE_DIR / "all_model_metrics.csv", index=False)
metrics[metrics.setting.ne("Full")].to_csv(TABLE_DIR / "topk_ridge_metrics.csv", index=False)
predictions.to_csv(TABLE_DIR / "predictions.csv", index=False)
metric_columns = ["Log_loss", "Brier", "AUROC", "AUPRC", "AP"]
macro = metrics[metrics.split.eq("test")].groupby(["method", "setting"], sort=False)[metric_columns].mean().reset_index()
macro.to_csv(TABLE_DIR / "unweighted_cohort_macro_average.csv", index=False)
weight_rows = []
for method, fit in fits.items():
    if "weights" in fit:
        w = np.asarray(fit["weights"])
        pd.DataFrame(dict(sample_id=cohorts[TRAIN]["sample_id"], pCR=y, weight=w)).to_csv(
            TABLE_DIR / (method.replace(" ", "_") + "_training_weights.csv"), index=False)
        weight_rows.append(dict(method=method, ESS=w.sum()**2 / (w @ w), maximum_share=w.max()/w.sum()))
pd.DataFrame(weight_rows).to_csv(TABLE_DIR / "weight_summary.csv", index=False)
display(metrics[metrics.setting.eq("Full") & metrics.split.eq("test")].pivot(index="cohort", columns="method", values="AUROC"))


In [ ]:
COLORS = {"LR": "#6B7C96", "Lasso": "#7565A8", "Elastic Net": "#5B9FB4",
          "SCAD": "#98A9DD", "DWR": "#7FB195", "SRDO": "#C06BA5",
          "SVI": "#E39BB5", "VBLR": "#C9A052", "DW-BLR": "#D87D64"}
MARKERS = dict(zip(METHODS, ["p", "s", "^", "v", "P", "X", "h", "D", "o"]))
STYLES = dict(zip(METHODS, ["--", "--", "-.", ":", "-.", "-.", "--", ":", "-"]))
METRICS = ["Log_loss", "Brier", "AUROC", "AUPRC"]
LABELS = {"Log_loss": "Log-loss ↓", "Brier": "Brier score ↓", "AUROC": "AUROC ↑", "AUPRC": "AUPRC ↑"}
DW_DISPLAY_ORDER = ["DEK", "NFIB", "FEN1", "CCNA2", "FBXO5",
                    "STMN1", "CTPS1", "GATA3", "CA12", "SLC7A8"]
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10,
    "axes.labelcolor": "#324156", "text.color": "#324156",
    "xtick.color": "#59677F", "ytick.color": "#59677F",
    "axes.edgecolor": "#C1CBD6", "axes.linewidth": .75, "axes.titleweight": "normal",
    "pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none",
    "figure.facecolor": "white", "savefig.facecolor": "white"})
saved_figures = {}

def finish_axes(ax, grid=False):
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_axisbelow(True)
    ax.tick_params(length=3, width=.7)
    if grid:
        ax.grid(axis="y", color="#EBEFF4", linewidth=.65)

def method_handles():
    return [Line2D([0], [0], color=COLORS[m], marker=MARKERS[m], linestyle=STYLES[m],
        markersize=4, markerfacecolor=COLORS[m] if m == "DW-BLR" else "white",
        linewidth=1.8 if m == "DW-BLR" else 1.3, label=m) for m in METHODS]

def save_figure(fig, name):
    files = []
    for extension in ("png", "pdf", "svg"):
        target = FIG_DIR / f"{name}.{extension}"
        temporary = FIG_DIR / f"{name}_{uuid4().hex}.tmp"
        fig.savefig(temporary, format=extension, dpi=300, bbox_inches="tight")
        try:
            temporary.replace(target)
        except OSError:
            target = FIG_DIR / f"{name}_{uuid4().hex[:8]}.{extension}"
            temporary.replace(target)
        files.append(target.name)
    saved_figures[name] = files
    if get_ipython() is not None:
        plt.show()
    plt.close(fig)

test_metrics = metrics[metrics.cohort.isin(TESTS)]


In [ ]:
base_limits = {
    "Log_loss": [(.56,.66),(.52,.62),(.35,.45),(.39,.49),(.38,.48),(.27,.37)],
    "Brier": [(.19,.23),(.17,.21),(.10,.14),(.12,.16),(.11,.15),(.07,.11)],
    "AUROC": [(.63,.83),(.59,.79),(.62,.82),(.68,.88),(.55,.75),(.72,.92)],
    "AUPRC": [(.41,.71),(.37,.67),(.07,.37),(.43,.73),(.15,.45),(.18,.48)],
}
fig, axes = plt.subplots(6, 4, figsize=(19.2, 20.8))
for i, cohort in enumerate(TESTS):
    for j, metric in enumerate(METRICS):
        ax = axes[i, j]
        frame = test_metrics[test_metrics.cohort.eq(cohort) & test_metrics.setting.ne("Full")]
        for method in METHODS:
            values = frame[frame.method.eq(method)].set_index("top_k").loc[list(TOP_K), metric]
            ax.plot(range(3), values, color=COLORS[method], marker=MARKERS[method],
                linestyle=STYLES[method], linewidth=1.8 if method == "DW-BLR" else 1.15,
                markersize=4, markerfacecolor=COLORS[method] if method == "DW-BLR" else "white")
        lo, hi = base_limits[metric][i]
        span = hi - lo
        ax.set_ylim(min(lo, float(frame[metric].min())-.05*span),
                    max(hi, float(frame[metric].max())+.05*span))
        ax.set_xticks(range(3), ["Top3", "Top5", "Top10"], fontsize=8)
        ax.set_xlim(-.25, 2.25)
        ax.tick_params(axis="y", labelsize=8)
        ax.yaxis.set_major_locator(MultipleLocator(.01 if metric == "Brier" else .02 if metric == "Log_loss" else .05))
        ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
        if i == 0:
            ax.set_title(LABELS[metric], fontsize=13, pad=24, fontweight="bold")
        if j == 0:
            ax.text(-.20, .56, cohort, transform=ax.transAxes, ha="right", fontsize=10, fontweight="bold")
            ax.text(-.20, .40, f"n = {len(cohorts[cohort]['y'])}",
                    transform=ax.transAxes, ha="right", fontsize=9, color="#8490A3")
        finish_axes(ax)
fig.legend(handles=method_handles(), loc="lower center", ncol=9, frameon=False,
           fontsize=9.5, bbox_to_anchor=(.55, .025), columnspacing=1.25, handlelength=2.4)
fig.subplots_adjust(left=.115, right=.985, bottom=.065, top=.95, hspace=.32, wspace=.24)
save_figure(fig, "topk_ridge_four_metrics_6x4")


In [ ]:
selected_dw = top_sets[("DW-BLR", 10)]
dw_genes = [g for g in DW_DISPLAY_ORDER if g in selected_dw] + [g for g in selected_dw if g not in DW_DISPLAY_ORDER]
union_genes = list(dict.fromkeys(g for m in METHODS for g in top_sets[(m, 10)]))
effects = gene_statistics(cohorts, genes, union_genes)
test_mask = effects.cohort.isin(TESTS)
dw_mask = test_mask & effects.gene.isin(dw_genes)
if not effects.fit_ok.all():
    raise RuntimeError("A univariable fit is not estimable; inspect it rather than assigning nonsignificance")
effects["p_bh_union_tests"] = np.nan
effects.loc[test_mask, "p_bh_union_tests"] = bh_adjust(effects.loc[test_mask, "p"])
effects["p_bh_dw10_tests"] = np.nan
effects.loc[dw_mask, "p_bh_dw10_tests"] = bh_adjust(effects.loc[dw_mask, "p"])
effects["rank_sum_p_bh_dw10_tests"] = np.nan
effects.loc[dw_mask, "rank_sum_p_bh_dw10_tests"] = bh_adjust(effects.loc[dw_mask, "rank_sum_p"])
training_direction = effects[effects.cohort.eq(TRAIN)].set_index("gene").beta
effects["training_beta"] = effects.gene.map(training_direction)
effects["direction_concordant"] = np.sign(effects.beta) == np.sign(effects.training_beta)
effects.to_csv(TABLE_DIR / "gene_effects_and_expression_tests.csv", index=False)
manifest["gene_analysis"] = dict(
    DW10_BH_family=int(dw_mask.sum()), comparison_BH_family=int(test_mask.sum()),
    display_order=dw_genes)

def stars(p):
    if pd.isna(p):
        return "NA"
    return "***" if p < .01 else "**" if p < .05 else "*" if p < .10 else "ns"


In [ ]:
preferred = ["SLC7A8", "GATA3", "CA12", "NFIB", "FBXO5",
             "CTPS1", "DEK", "STMN1", "CCNA2", "FEN1"]
correlation_order = [g for g in preferred if g in dw_genes] + [g for g in dw_genes if g not in preferred]
ordered = pd.DataFrame(X[:, [genes.index(g) for g in correlation_order]], columns=correlation_order).corr()
ordered.to_csv(TABLE_DIR / "DWBLR_Top10_training_correlation.csv")
fig, ax = plt.subplots(figsize=(8.8, 7.15))
im = ax.imshow(ordered, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(ordered)), ordered.columns, rotation=45, ha="right", fontstyle="italic", fontsize=13)
ax.set_yticks(range(len(ordered)), ordered.index, fontstyle="italic", fontsize=13)
for i in range(len(ordered)):
    for j in range(len(ordered)):
        value = ordered.iloc[i, j]
        ax.text(j, i, f"{value:.2f}", ha="center", va="center",
                color="white" if abs(value) > .65 else "#324156", fontsize=11)
for spine in ax.spines.values():
    spine.set_color("#C1CBD6")
bar = fig.colorbar(im, ax=ax, shrink=.90, pad=.055, ticks=np.arange(-1,1.01,.25))
bar.set_label("Pearson correlation", fontsize=12, labelpad=9)
bar.ax.tick_params(labelsize=11)
fig.tight_layout()
save_figure(fig, "DWBLR_Top10_training_correlation")


In [ ]:
rd_color, pcr_color = "#6AA8B9", "#DF8A70"
jitter = np.random.default_rng(parameters.SEED)
expression = {}
for cohort in TESTS:
    ex = cohorts[cohort]["X"][:, [genes.index(g) for g in dw_genes]]
    expression[cohort] = (ex - ex.mean(axis=0)) / ex.std(axis=0, ddof=0)
ymin = min(-4.2, min(float(ex.min()) for ex in expression.values())-.2)
bracket_y = max(5.7, max(float(ex.max()) for ex in expression.values())+.7)
fig, axes = plt.subplots(2, 3, figsize=(18, 8.83), sharey=True)
expression_rows = []
for ax, cohort in zip(axes.flat, TESTS):
    data, ex = cohorts[cohort], expression[cohort]
    yy = data["y"]
    for j, gene in enumerate(dw_genes):
        for outcome, color, offset in ((0, rd_color, -.19), (1, pcr_color, .19)):
            values = ex[yy == outcome, j]
            ax.boxplot([values], positions=[j+offset], widths=.30, patch_artist=True,
                manage_ticks=False, showfliers=False,
                medianprops=dict(color=color, linewidth=1.4),
                whiskerprops=dict(color=color, linewidth=.8), capprops=dict(color=color, linewidth=.8),
                boxprops=dict(facecolor=color, edgecolor=color, alpha=.25))
            ax.scatter(j+offset+jitter.uniform(-.11,.11,len(values)), values,
                       s=4, color=color, alpha=.4, linewidths=0)
        pval = effects.loc[effects.cohort.eq(cohort) & effects.gene.eq(gene), "rank_sum_p_bh_dw10_tests"].iloc[0]
        ax.plot([j-.19,j-.19,j+.19,j+.19], [bracket_y-.1,bracket_y,bracket_y,bracket_y-.1],
                color="#9DA9B9", linewidth=.6)
        ax.text(j, bracket_y+.12, stars(pval), ha="center", va="bottom",
                color="#8490A3" if pval >= .1 else "#324156", fontsize=8)
        expression_rows.extend(dict(cohort=cohort, sample_id=sid, gene=gene, pCR=int(label),
            standardized_expression=float(value)) for sid,label,value in zip(data["sample_id"],yy,ex[:,j]))
    ax.set_xticks(range(10), dw_genes, rotation=48, ha="right", fontstyle="italic", fontsize=9)
    ax.set_title(cohort, loc="left", fontsize=11, pad=13)
    ax.text(1, 1.04, f"pCR, n = {int(yy.sum())} | RD, n = {int((yy == 0).sum())}",
            transform=ax.transAxes, ha="right", fontsize=8, color="#8490A3")
    ax.set_ylim(ymin, bracket_y+.75)
    ax.set_yticks(np.arange(np.ceil(ymin),bracket_y,1))
    finish_axes(ax, grid=True)
for ax in axes[:,0]:
    ax.set_ylabel("Standardized expression", fontsize=10)
fig.legend(handles=[Patch(facecolor=rd_color, edgecolor=rd_color, alpha=.45, label="RD"),
                    Patch(facecolor=pcr_color, edgecolor=pcr_color, alpha=.45, label="pCR")],
           loc="lower center", bbox_to_anchor=(.5,.055), ncol=2, frameon=False)
fig.text(.5,.018,r"BH-adjusted $P$:  * < 0.10    ** < 0.05    *** < 0.01    ns ≥ 0.10",
         ha="center", fontsize=9, color="#66768D")
fig.subplots_adjust(left=.055, right=.99, top=.91, bottom=.20, hspace=.36, wspace=.17)
pd.DataFrame(expression_rows).to_csv(TABLE_DIR / "DWBLR_Top10_expression_plot_data.csv", index=False)
save_figure(fig, "DWBLR_Top10_expression_2x3")


In [ ]:
beta_matrix = effects[effects.gene.isin(dw_genes)].pivot(index="gene",columns="cohort",values="beta").loc[dw_genes,list(cohorts)]
q_matrix = effects[effects.gene.isin(dw_genes)].pivot(index="gene",columns="cohort",values="p_bh_dw10_tests").loc[dw_genes,list(cohorts)]
beta_matrix.to_csv(TABLE_DIR / "DWBLR_Top10_effect_matrix.csv")
q_matrix.to_csv(TABLE_DIR / "DWBLR_Top10_BH_adjusted_P_matrix.csv")
effect_cmap = LinearSegmentedColormap.from_list("effect_context",
    ["#477899","#A8C4D4","#F9F8F7","#E9B29F","#B65B46"])
limit = max(2., float(np.nanmax(np.abs(beta_matrix.to_numpy()))))
fig = plt.figure(figsize=(17.8,8.2))
train_ax = fig.add_axes([.065,.21,.095,.65])
test_ax = fig.add_axes([.186,.21,.575,.65])
bx = fig.add_axes([.787,.21,.19,.65])
for ax, columns in ((train_ax,[TRAIN]),(test_ax,TESTS)):
    image_values = beta_matrix[columns].to_numpy()
    im = ax.imshow(image_values, cmap=effect_cmap, vmin=-limit, vmax=limit, aspect="auto")
    ax.set_xticks([])
    ax.set_yticks(range(10), dw_genes if ax is train_ax else [""]*10,
                  fontstyle="italic", fontweight="bold", fontsize=11, color="#111111")
    ax.tick_params(which="both", length=0, pad=9)
    for side in ax.spines.values():
        side.set_visible(False)
    for j, cohort in enumerate(columns):
        ax.text(j,1.105,cohort,transform=ax.get_xaxis_transform(),ha="center",fontsize=11,fontweight="bold")
        ax.text(j,1.035,f"n = {len(cohorts[cohort]['y'])} · pCR = {int(cohorts[cohort]['y'].sum())}",
                transform=ax.get_xaxis_transform(),ha="center",fontsize=8.5,color="#75849A")
        for i, gene in enumerate(dw_genes):
            ax.text(j,i if cohort == TRAIN else i-.12,f"{beta_matrix.loc[gene,cohort]:+.2f}",
                    ha="center",va="center",fontsize=10.5)
            if cohort != TRAIN:
                label = stars(q_matrix.loc[gene,cohort])
                ax.text(j,i+.24,"" if label=="ns" else label,ha="center",va="center",fontsize=8.5)
    ax.set_xticks(np.arange(-.5,len(columns),1),minor=True)
    ax.set_yticks(np.arange(-.5,10,1),minor=True)
    ax.grid(which="minor",color="white",linewidth=1.4)
replication_rows = []
bar_colors = ["#B4C5D4","#91A5D8","#D27F68"]
for i, gene in enumerate(dw_genes):
    sub = effects[effects.gene.eq(gene) & effects.cohort.isin(TESTS)]
    concordant = sub.direction_concordant
    counts = [int(concordant.sum()),int((concordant & (sub.p_bh_dw10_tests < .10)).sum()),
              int((concordant & (sub.p_bh_dw10_tests < .05)).sum())]
    replication_rows.append(dict(gene=gene,same_direction=counts[0],
        same_direction_p_lt_010=counts[1],same_direction_p_lt_005=counts[2],estimable=len(sub)))
    if i % 2 == 0:
        bx.axhspan(i-.5,i+.5,color="#F5F7FA",zorder=0)
    for offset,value,color in zip((-.23,0,.23),counts,bar_colors):
        bx.barh(i+offset,value,height=.185,color=color,edgecolor="none",zorder=3)
        bx.text(value+.09,i+offset,str(value),fontsize=9,va="center",color="#53637B")
bx.set(ylim=(9.5,-.5),xlim=(0,6.65))
bx.set_yticks([])
bx.set_xticks(range(0,7,2))
bx.text(.5,1.105,"Effect Direction Concordance",transform=bx.transAxes,
        ha="center",fontsize=11,fontweight="bold")
bx.set_xlabel("Cohorts (out of 6)",loc="right",fontsize=9.5)
finish_axes(bx)
bx.spines["left"].set_visible(False)
bx.grid(axis="x",color="#EAF0F5",linewidth=.6)
bx.legend(handles=[Patch(color=c,label=t) for c,t in zip(bar_colors,
    ["Same sign",r"Same sign ($P$ < 0.10)",r"Same sign ($P$ < 0.05)"])],
    loc="upper left",bbox_to_anchor=(0,-.13),frameon=False,fontsize=8.5,borderaxespad=0)
cax = fig.add_axes([.21,.105,.39,.022])
fig.colorbar(im,cax=cax,orientation="horizontal",
    label="Univariate logistic β per within-cohort SD (pCR = 1)")
cax.tick_params(labelsize=8.5)
fig.text(.075,.165,r"* $P$ < 0.10   ** $P$ < 0.05   *** $P$ < 0.01 (BH-adjusted P-value)",fontsize=9)
pd.DataFrame(replication_rows).to_csv(TABLE_DIR / "DWBLR_Top10_direction_replication.csv",index=False)
save_figure(fig,"DWBLR_Top10_effect_heatmap")


In [ ]:
other_order = ["SKA1","EPHB3","RAP2A","PLIN2","ELF5","ANP32B","CDC45","GMNN","ILF2",
               "MCM2","SERBP1","SFRP1","ROGDI","CHST15","GLI3","KCNK15","MLPH","SLC22A5","TSPAN1"]
priority = {gene:i for i,gene in enumerate(dw_genes+other_order)}
forest_order = sorted(union_genes,key=lambda g:(training_direction[g] < 0,g not in dw_genes,
                                               priority.get(g,len(priority)+union_genes.index(g))))
positive, negative, nonsignificant = "#E3894B","#568CB8","#7CAA91"
forest_data = effects[effects.cohort.isin(TESTS)].copy()
forest_data["plot_row"] = forest_data.gene.map({g:i for i,g in enumerate(forest_order)})
forest_data.to_csv(TABLE_DIR / "Top10_union_forest_plot_data.csv",index=False)
lower_limit = min(-4.,float(np.floor(forest_data.lower.min())))
upper_limit = max(3.,float(np.ceil(forest_data.upper.max())))
n_positive = sum(training_direction[g] >= 0 for g in forest_order)
fig, axes = plt.subplots(1,6,figsize=(22,8.8),sharey=True)
for ax, cohort in zip(axes,TESTS):
    frame = forest_data[forest_data.cohort.eq(cohort)].set_index("gene").loc[forest_order]
    for j,(gene,row) in enumerate(frame.iterrows()):
        if j % 2 == 0:
            ax.axhspan(j-.5,j+.5,color="#F5F7FA",zorder=0)
        color = (positive if row.beta > 0 else negative) if row.p_bh_union_tests < .10 else nonsignificant
        ax.errorbar(row.beta,j,xerr=[[row.beta-row.lower],[row.upper-row.beta]],
            color=color,fmt="o" if gene in dw_genes else "s",markersize=3.7,
            markeredgecolor="white",markeredgewidth=.35,capsize=1.8,linewidth=.9,zorder=3)
    ax.axvline(0,color="#A7B4C4",linestyle="--",linewidth=.8)
    ax.axhline(n_positive-.5,color="#CBD5E0",linewidth=.7)
    ax.set_title(cohort,fontsize=11,pad=26,fontweight="bold")
    data = cohorts[cohort]
    ax.text(.5,1.018,f"n = {len(data['y'])} · pCR = {int(data['y'].sum())}",
            transform=ax.transAxes,fontsize=8,color="#7C8AA0",ha="center")
    finish_axes(ax)
    ax.grid(axis="x",color="#EAF0F5",linewidth=.5)
    ax.set(ylim=(len(forest_order)-.4,-.6),xlim=(lower_limit-.15,upper_limit+.15))
    ax.set_xticks(np.arange(lower_limit,upper_limit+1,1))
    ax.tick_params(axis="x",labelsize=8)
axes[0].set_yticks(range(len(forest_order)),forest_order,fontsize=9,fontstyle="italic")
for label,gene in zip(axes[0].get_yticklabels(),forest_order):
    if gene in dw_genes:
        label.set_color("#BF5755")
        label.set_fontweight("bold")
fig.supxlabel("Log odds ratio for pCR (95% CI)",y=.125,fontsize=11)
fig.legend(handles=[
    Line2D([0],[0],color=positive,label="Significantly higher pCR odds"),
    Line2D([0],[0],color=negative,label="Significantly lower pCR odds"),
    Line2D([0],[0],color=nonsignificant,label="Not statistically significant"),
    Line2D([0],[0],color="#6B7C96",marker="o",linestyle="none",label="DW-BLR Top 10"),
    Line2D([0],[0],color="#6B7C96",marker="s",linestyle="none",label="Other genes")],
    loc="lower center",ncol=5,frameon=False,bbox_to_anchor=(.53,.06),fontsize=9)
fig.text(.53,.022,r"Statistical significance was defined as BH-adjusted $P$ < 0.1.",
         ha="center",fontsize=8,color="#7C8AA0")
fig.subplots_adjust(left=.065,right=.99,top=.89,bottom=.20,wspace=.17)
save_figure(fig,"Top10_union_univariable_forest_1x6")


In [ ]:
replication = []
for method in METHODS:
    for gene in top_sets[(method,10)]:
        frame = effects[effects.gene.eq(gene) & effects.cohort.isin(TESTS)]
        same, significant = frame.direction_concordant, frame.p_bh_union_tests < .10
        replication.append(dict(method=method,gene=gene,
            concordant_significant=int((same & significant).sum()),
            concordant_nonsignificant=int((same & ~significant).sum()),discordant=int((~same).sum())))
replication = pd.DataFrame(replication)
replication.to_csv(TABLE_DIR / "method_Top10_direction_replication.csv",index=False)
groups = [(m,m) for m in METHODS]
if set(top_sets[("Lasso",10)]) == set(top_sets[("SCAD",10)]):
    groups = [("Lasso / SCAD" if m=="Lasso" else m,m) for m in METHODS if m!="SCAD"]
nrows,ncols = (2,4) if len(groups)==8 else (3,3)
fig, axes = plt.subplots(nrows,ncols,figsize=(19.5,10.5 if nrows==2 else 14.7))
categories = ["concordant_significant","discordant","concordant_nonsignificant"]
rep_colors = {"concordant_significant":"#D27F68","discordant":"#6683AC","concordant_nonsignificant":"#BFC6D0"}
for ax,(label,method) in zip(axes.flat,groups):
    display_genes = dw_genes if method=="DW-BLR" else top_sets[(method,10)]
    frame = replication[replication.method.eq(method)].set_index("gene").loc[display_genes]
    bottom = np.zeros(10)
    for category in categories:
        heights = frame[category].to_numpy()/len(TESTS)*100
        ax.bar(range(10),heights,bottom=bottom,width=.58,
               color=rep_colors[category],edgecolor="none",linewidth=0)
        bottom += heights
    np.testing.assert_allclose(bottom,100)
    ax.set_xticks(range(10),frame.index,rotation=55,ha="right",fontstyle="italic",fontsize=9)
    ax.set_yticks([0,25,50,75,100])
    ax.yaxis.set_major_formatter(PercentFormatter())
    ax.set(ylim=(0,102),xlim=(-.65,9.65))
    ax.set_title(label,fontsize=12,fontweight="normal",
                 color="black" if method=="DW-BLR" else "#324156",pad=14)
    finish_axes(ax,grid=True)
for ax in axes[:,0]:
    ax.set_ylabel("Percentage of validation cohorts",fontsize=10)
fig.legend(handles=[Patch(color=rep_colors[k],label=t) for k,t in zip(
    ["concordant_significant","concordant_nonsignificant","discordant"],
    [r"Directionally concordant, BH-adjusted $P$ < 0.10",
     r"Directionally concordant, BH-adjusted $P$ ≥ 0.10","Directionally discordant"])],
    loc="lower center",ncol=3,frameon=False,fontsize=10,bbox_to_anchor=(.5,.02))
fig.subplots_adjust(left=.055,right=.99,top=.93,bottom=.16 if nrows==2 else .13,
                    hspace=.50,wspace=.25)
save_figure(fig,"method_Top10_direction_replication")


In [ ]:
roc_cohorts = [TRAIN,"GSE163882","GSE41998","GSE230881","GSE140494","GSE194040"]
roc_groups = ["Overall","HR+/HER2-","TNBC"]
roc_colors = ["#D38167","#5E99AB","#6A5C95"]
roc_styles = ["-",(0,(5,2.5)),(0,(4,2,1,2))]
patients = predictions[predictions.method.eq("DW-BLR") & predictions.setting.eq("Top10")
                       & predictions.cohort.isin(roc_cohorts)].merge(
    clinical[["cohort","sample_id","subgroup","HR_by_ESR1_fallback"]],
    on=["cohort","sample_id"],validate="one_to_one",how="left")
if patients.subgroup.isna().any():
    raise ValueError("Missing clinical subgroup labels after matching sample IDs")
fig, axes = plt.subplots(2,3,figsize=(16.5,11.8))
fig.subplots_adjust(left=.063,right=.986,top=.95,bottom=.14,wspace=.24,hspace=.60)
roc_summary, roc_coordinates = [], []
for ax, cohort in zip(axes.flat,roc_cohorts):
    frame = patients[patients.cohort.eq(cohort)]
    ax.set_aspect("equal",adjustable="box")
    ax.plot([0,1],[0,1],linestyle=(0,(4,4)),linewidth=1,color="#BAC5D0",zorder=0)
    legend_columns = []
    for group,color,style in zip(roc_groups,roc_colors,roc_styles):
        sub = frame if group=="Overall" else frame[frame.subgroup.eq(group)]
        yy, pp = sub.pCR.to_numpy(int), sub.probability.to_numpy(float)
        estimable = len(yy)>0 and np.unique(yy).size==2
        value = np.nan
        if estimable:
            fpr,tpr,threshold = roc_curve(yy,pp,drop_intermediate=False)
            value = roc_auc_score(yy,pp)
            np.testing.assert_allclose(auc(fpr,tpr),value,rtol=0,atol=1e-14)
            ax.plot(fpr,tpr,color=color,linestyle=style,linewidth=2,zorder=3)
            roc_coordinates.append(pd.DataFrame(dict(cohort=cohort,group=group,
                FPR=fpr,TPR=tpr,probability_threshold=threshold)))
        roc_summary.append(dict(cohort=cohort,group=group,n=len(yy),n_pCR=int(yy.sum()),
            n_RD=int(len(yy)-yy.sum()),AUROC=value,estimable=estimable))
        swatch = DrawingArea(24,10,0,0)
        swatch.add_artist(Line2D([0,24],[5,5],color=color,linestyle=style,linewidth=2))
        heading = HPacker(children=[swatch,TextArea(group,textprops=dict(size=9.8,color="#2E3D52"))],
                          align="center",pad=0,sep=5)
        details = VPacker(children=[
            TextArea(f"AUROC = {value:.3f}" if estimable else "AUROC = NA",textprops=dict(size=7.8,color="#7C858F")),
            TextArea(f"n = {len(yy)}; pCR = {int(yy.sum())}",textprops=dict(size=7.8,color="#7C858F"))],
            align="center",pad=0,sep=2)
        legend_columns.append(VPacker(children=[heading,details],align="center",pad=0,sep=5))
    legend = AnchoredOffsetbox(loc="upper center",
        child=HPacker(children=legend_columns,align="top",pad=0,sep=18),frameon=False,
        bbox_to_anchor=(.5,-.19),bbox_transform=ax.transAxes,borderpad=0,pad=0)
    ax.add_artist(legend)
    ax.set_title(cohort,loc="left",pad=15,fontsize=13,fontweight="bold")
    ax.set(xlim=(-.01,1.01),ylim=(-.01,1.01))
    ticks = np.linspace(0,1,6)
    ax.set_xticks(ticks,[f"{v:.1f}" for v in ticks])
    ax.set_yticks(ticks,[f"{v:.1f}" for v in ticks])
    ax.set_xlabel("False positive rate (1 - specificity)",labelpad=7,fontsize=10.5)
    ax.set_ylabel("True positive rate (sensitivity)",labelpad=7,fontsize=10.5)
    finish_axes(ax)
    ax.grid(color="#EDF1F5",linewidth=.7)
pd.DataFrame(roc_summary).to_csv(TABLE_DIR / "subgroup_AUROC_summary.csv",index=False)
pd.concat(roc_coordinates,ignore_index=True).to_csv(TABLE_DIR / "subgroup_ROC_coordinates.csv",index=False)
patients.to_csv(TABLE_DIR / "subgroup_patient_predictions.csv",index=False)
manifest["subgroup_analysis"] = dict(cohorts=roc_cohorts,groups=roc_groups,
    clinical_file=str(SUBGROUP_FILE.relative_to(ROOT)),clinical_sha256=identity["clinical_subgroups_sha256"],
    clinical_source_provenance=clinical_provenance,
    GSE230881_unclassified_n=int(((patients.cohort=="GSE230881") & (patients.subgroup=="Unclassified HR")).sum()),
    training_ESR1_fallback_n=int(clinical.loc[clinical.cohort.eq(TRAIN),"HR_by_ESR1_fallback"].astype(str).str.lower().eq("true").sum()))
save_figure(fig,"DWBLR_subgroup_ROC_2x3")


In [ ]:
expected_figures = {
    "topk_ridge_four_metrics_6x4","DWBLR_Top10_effect_heatmap",
    "Top10_union_univariable_forest_1x6","method_Top10_direction_replication",
    "DWBLR_Top10_expression_2x3","DWBLR_subgroup_ROC_2x3","DWBLR_Top10_training_correlation",
}
if set(saved_figures) != expected_figures:
    raise RuntimeError("Unexpected set of generated figures")
manifest.update(status="complete_with_convergence_warnings" if not convergence.converged.all() else "complete",
    nonconverged_methods=convergence.loc[~convergence.converged,"method"].tolist(),
    n_metric_rows=len(metrics),n_prediction_rows=len(predictions),
    tables=[str(p.relative_to(TABLE_DIR)) for p in sorted(TABLE_DIR.rglob("*.csv"))],
    top_analysis_figures=saved_figures,
    figures=[f for files in saved_figures.values() for f in files if f.endswith(".png")])
save_json(RUN_DIR / "manifest.json",manifest)
print(RUN_DIR)
